# 3-5 loc与iloc数据选择

`loc` 和 `iloc` 都可以按“行、列”选择数据，主要区别是使用标签还是整数位置。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

In [ ]:
df = pd.read_csv(data_dir / "sample.csv", dtype={"StockCode": "string"})
df.head()

商品编码既可能含字母，也可能有前导零，适合按字符串读取。

## 2. 对比 loc 与 iloc

| 对比项 | `loc` | `iloc` |
|---|---|---|
| 依据 | 索引标签和列名 | 从 0 开始的整数位置 |
| 切片终点 | 包含终点标签 | 不包含终点位置 |
| 示例 | `df.loc[:3, "StockCode"]` | `df.iloc[:3, 1]` |
| 常见场景 | 按字段和条件筛选 | 按第几行、第几列筛选 |

两者通用形式为 `df.方法[行选择, 列选择]`；`:` 表示全部。

## 3. 使用 loc

### 3.1 指定行与列

当前索引为 `0, 1, 2, ...`，所以 `:3` 包含标签 3，共返回前 4 行。列名列表可以同时选择多列。

In [ ]:
df.loc[:3, ["StockCode", "Description", "Quantity"]]

### 3.2 单列 Series 与单列 DataFrame

单个列名返回 Series；包含一个列名的列表返回 DataFrame。

In [ ]:
df.loc[:3, "StockCode"]

In [ ]:
df.loc[:3, ["StockCode"]]

### 3.3 按条件筛选

编码 `71053` 在这里是字符串，因此比较时要带引号。

In [ ]:
df.loc[df["StockCode"] == "71053"]

In [ ]:
df.dtypes

整数索引还可以用于构造布尔条件；下例返回索引为偶数的行。

In [ ]:
df.loc[df.index % 2 == 0].head()

多条件组合使用 `&` 表示“并且”，`|` 表示“或者”；每个条件都应加括号。

In [ ]:
df.loc[(df["Quantity"] >= 6) & (df["UnitPrice"] < 3), ["StockCode", "Quantity", "UnitPrice"]].head()

## 4. 使用 iloc

`iloc[:3, :]` 选择第 1～3 行；`iloc[:3, :3]` 再限制到前 3 列。

In [ ]:
df.iloc[:3, :]

In [ ]:
df.iloc[:3, :3]

即使索引换成文字，`iloc` 仍按位置选择。下面明确展示切片终点的区别。

In [ ]:
demo = pd.DataFrame({"值": [10, 20, 30, 40]}, index=["a", "b", "c", "d"])
demo.loc["a":"c"]

In [ ]:
demo.iloc[0:2]

前者包含 `c`，共 3 行；后者只包含位置 0、1，共 2 行。

## 5. 安全地修改选中的数据

通过一次 `loc[条件, 列名]` 赋值可以明确修改哪张表。不要使用 `df[条件]["列名"] = ...` 这样的链式赋值。

In [ ]:
edited = df.copy()
edited.loc[edited["StockCode"] == "71053", "Quantity"] = 10
edited.loc[edited["StockCode"] == "71053", ["StockCode", "Quantity"]]

## 6. 要点总结

- `loc` 按标签选择，标签切片包含终点；`iloc` 按位置选择，不包含终点。
- 单列名返回 Series，列名列表返回 DataFrame。
- 比较时检查字段类型；多个条件用括号配合 `&`、`|`。
- 修改数据用一次 `loc` 赋值，避免链式赋值；旧版 `.ix` 不再使用。